## csv_to_dataframe: conversion from CSV EMG data to a trial-level Pandas DataFrame

This notebook takes one session's 1000 Hz EMG export (`P#_S#_EMG1kHz.csv` or `P#_S#_EMG1kHz_filt.csv`) and turns it into a single **trial-level** DataFrame: one row per trial, with the full EMG recording for that trial stored as an array in a column, plus the task name and channel names attached.

**Why this shape?** The raw CSV very tall matrix of one row per timepoint per trial, which can make analysis a challenge. Code used for plotting, feature extraction, classification expects one row per trial instead, so this notebook does that reshaping once.

**What you need before running this:**
- `P#_S#_EMG1kHz.csv` or `P#_S#_EMG1kHz_filt.csv` — the 1000 Hz EMG samples for one session
- `P#_S#_meta.json` — trial-level metadata for that same session (`TrialID`, `TaskNumber`, `TrialNumber`, `RestTime`, `HoldTime`)
- `P#_metadata.json` — channel name lookup for that participant
- `movements.json` — dataset-wide task number → task name lookup

The cells below default to the files in `sample_set/`, so you can run this notebook top to bottom with no changes to see how it works, then swap in your own session's paths in **Step 0**.

## Step 0: Setup

Only `numpy` and `pandas` are needed for this script. If you are working in a Python environment created using the `miRPNI.yaml` file in the code repo, then you should be good to go. If you are using a different Python setup and don't have these packages yet, uncomment the line below.

In [ ]:
# !pip install numpy pandas

import numpy as np
import pandas as pd

Point these filepaths at the session you want to process. The defaults below load the sample session shipped in this repo.

In [ ]:
CSV_PATH = "sample_set/csv/P1_S12_EMG1kHz.csv"
META_PATH = "sample_set/meta/P1_S12_meta.json"
CHANNELS_PATH = "sample_set/meta/P1_metadata.json"
TASKS_PATH = "sample_set/movements.json"

## Step 1: Load trial-level metadata

`P#_S#_meta.json` already has exactly one row per trial (`TrialID`, `TaskNumber`, `TrialNumber`, `RestTime`, `HoldTime`). This dataframe is the basis for everything else we'll be adding in the later cell blocks.

In [ ]:
trial_meta = pd.read_json(META_PATH)
trial_meta.head()

## Step 2: Reshape the raw EMG CSV into one array per trial

The raw CSV is long-format: every row is a single timepoint for a single trial, with one column per EMG channel (`EMG1k_1`, `EMG1k_2`, ...) plus a `TrialID` column tying each row back to a trial.

To get one `(numSamples, numChannels)` array per trial, we group the rows by `TrialID` and pull out just the channel columns for each group.

In [ ]:
emg_raw = pd.read_csv(CSV_PATH)
emg_raw.head()

In [ ]:
channel_cols = [c for c in emg_raw.columns if c.startswith("EMG1k_")]

emg_arrays = {
    tid: group[channel_cols].to_numpy()
    for tid, group in emg_raw.groupby("TrialID")
}

trial_meta["EMG1k"] = trial_meta["TrialID"].map(emg_arrays)

Sanity check: each trial's array should be `(numSamples, numChannels)` — for the sample session that's 8 seconds at 1 kHz across 8 channels, so `(8000, 8)`.

In [ ]:
trial_meta["EMG1k"].iloc[0].shape

## Step 3: Attach human-readable task names

`trial_meta` only has a numeric `TaskNumber` for each trial. The metadata file `movements.json` maps every `TaskNumber` in the dataset to its `TaskName`, which we will add as another field in the dataframe.

In [ ]:
tasks = pd.read_json(TASKS_PATH)
tasks["TaskNumber"] = tasks["TaskNumber"].astype(int)
trial_meta["TaskNumber"] = trial_meta["TaskNumber"].astype(int)

trial_meta = trial_meta.merge(tasks[["TaskNumber", "TaskName"]], on="TaskNumber", how="left")
trial_meta[["TrialID", "TaskNumber", "TaskName"]].head()

## Step 4: Map channel numbers to names

Participant channel name (which electrode number is which muscle being recorded) is constant across every trial in a session, so this is a lookup dictionary rather than something we merge row-by-row into `trial_meta`. Use it whenever you need to label a column of `EMG1k` by muscle name.

In [ ]:
channels = pd.read_json(CHANNELS_PATH)
channel_names = channels.set_index("channelNumber")["channelName"].to_dict()
channel_names

## Result

`trial_meta` now has one row per trial, with the full EMG recording and task name attached. This is the DataFrame you'd hand off to plotting or feature-extraction code.

In [ ]:
trial_meta.head()